# 02 — Build graph & clean chargers

Loads the cached graph and raw chargers (wider central-Warsaw inventory), snaps each charger to its nearest road node, drops chargers outside the polygon (snap distance > `MAX_SNAP_DISTANCE_M`), imputes missing port counts, and writes the cleaned table to `data/processed/chargers_clean.csv`.

Per-step diagnostics are printed by `clean_chargers`:
- raw rows downloaded
- after coordinate cleaning
- after port imputation
- after snap-distance boundary filter
- after duplicate-node aggregation
- final station count and total port count

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve().parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

import pandas as pd
from src import config, graph_utils, charger_data

print(f'STUDY_AREA_MODE        = {config.STUDY_AREA_MODE}')
print(f'MAX_SNAP_DISTANCE_M    = {config.MAX_SNAP_DISTANCE_M}')
print(f'raw charger CSV path   = {config.RAW_CHARGER_CSV}')
print(f'clean charger CSV path = {config.PROCESSED_CHARGER_CSV}')

In [ ]:
G = graph_utils.get_or_build_graph()
print(f'graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

In [ ]:
df_raw = charger_data.load_chargers(prefer_cached=True)
print(f'raw rows: {len(df_raw):,}')
df_raw.head()

In [ ]:
df_clean = charger_data.clean_chargers(df_raw, G)
charger_data.save_clean_chargers(df_clean)
df_clean.head()

### Sanity check
Map of cleaned chargers (lon/lat scatter on top of all graph nodes).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter([G.nodes[n]['x'] for n in G.nodes], [G.nodes[n]['y'] for n in G.nodes], s=0.5, alpha=0.3, label='graph nodes')
if not df_clean.empty:
    ax.scatter(df_clean['longitude'], df_clean['latitude'], s=20, c='red', label='chargers (cleaned)')
ax.set_xlabel('lon'); ax.set_ylabel('lat'); ax.legend()
ax.set_title(f'Central Warsaw road nodes & cleaned chargers (mode={config.STUDY_AREA_MODE})')
plt.show()